# 22 · Told what the client reads: the slice QP, the loop filter, I or P

Notebook 19's model was told x264's mean QP and only ever saw x264's frames. Notebook 21 made the pairs again with what
the client can read off every frame (desktop-streamer, `stream-h264.js`) and with both of the host's lines: the slice
QP, whether the loop filter runs (NVENC: yes, the x264 line: no), and whether it is an I frame. Here notebook 19's body
and recipe are trained on them three ways:

1. **Blind** - told nothing.
2. **Told the QP** - the slice QP / 51.
3. **Told the QP and the frame** - the slice QP / 51, the loop filter (0 or 1), I frame (0 or 1).

All three ship the same way: the conditions are a straight line into the biases, `b + W c`, so the client computes 72
numbers a frame from what it read and the graph is the blind one's.

Two models are scored beside them without training: **notebook 19's**, told the slice QP where it learned on the mean -
what shipping it today would give - and **notebook 8's graph**, which ships now. Scores are split by encoder and frame
type, then by slice QP bin.

**The ruler.** The two told models are trained at two seeds; notebook 19's two seeds of one model landed at most 0.0004
SSIM-Y apart a bin.

**Limits.** Targets are video only. The NVENC frames were shrunk by swscale, not on the GPU as the host's are.

Needs: notebook 21 (`qp_slice_pairs/`), notebook 19 (`checkpoints/upscale_qp.pt`) and notebook 8's graph in
`dist/upscale/`. Writes each training to `checkpoints/qp_slice_runs/` and the best told model to
`checkpoints/upscale_qp_slice.pt`.

In [ ]:
"""Settings."""

import os

import numpy as np
import onnx
import torch
import torch.nn as nn
from onnx import numpy_helper
from torch.utils.data import DataLoader, Dataset, Subset

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import metrics, models, patches, sources, training, webexport
from helpers.training import DEVICE

PAIRS_DIR = sources.ROOT / "upscale" / "qp_slice_pairs"                    # notebook 21
RUN_DIR = training.CHECKPOINT_DIR / "qp_slice_runs"                        # each training, saved every epoch
CHECKPOINT = training.CHECKPOINT_DIR / "upscale_qp_slice.pt"               # the best told model
NOTEBOOK_19 = training.CHECKPOINT_DIR / "upscale_qp.pt"                    # told x264's mean QP
SHIPPED = webexport.MODELS_DIR / "upscale_screens_24x2_bpp0.46_tile320x180.onnx"   # notebook 8's, as it ships

QP_EDGES = (0, 18, 26, 30, 34, 38, 42, 46, 52)   # bins of the slice QP
QP_MAX = 51
CONDITIONS = ("qp", "deblocked", "intra")        # what a patch carries, in this order
WIDTH, BLOCKS, STEM = 24, 2, 3                   # notebook 8's body
BATCH_SIZE, LEARNING_RATE, MAX_EPOCHS = 32, 1e-3, 500
RUNS = {"blind": ((), (0,)),                                      # name: (conditions it is told, seeds)
        "told QP": (("qp",), (0, 1)),
        "told QP + frame": (("qp", "deblocked", "intra"), (0, 1))}

training.seed_everything(0)

## 1. Data

Every patch carries its three conditions, the QP divided by 51; a model takes the ones it is told and drops the rest.

In [ ]:
manifest = training.load_manifest(PAIRS_DIR)
training.describe(manifest)
SCALE = manifest["scale"]
ENCODERS = manifest["encoders"]
trim = training.trimmer(manifest)


class WithConditions(Dataset):
    """A split's pairs as ((LR, conditions (3, 1, 1)), HR), and each patch's labels as arrays."""

    def __init__(self, split, augment):
        self.pairs = patches.PatchPairs(PAIRS_DIR, split, manifest, augment=augment)
        files = manifest["splits"][split]
        self.labels = {name: np.load(PAIRS_DIR / files[name]) for name in ("qp", "deblocked", "intra", "encoder")}
        values = np.stack([self.labels["qp"] / QP_MAX, self.labels["deblocked"], self.labels["intra"]], axis=1)
        self.conditions = torch.from_numpy(values.astype(np.float32))[:, :, None, None]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        lr, hr = self.pairs[index]
        return (lr, self.conditions[index]), hr

    def where(self, encoder=None, intra=None, low=0, high=99):
        """The indices of the patches of one encoder and frame type in a slice QP range."""
        keep = (self.labels["qp"] >= low) & (self.labels["qp"] < high)
        if encoder is not None:
            keep &= self.labels["encoder"] == ENCODERS.index(encoder)
        if intra is not None:
            keep &= self.labels["intra"] == intra
        return np.flatnonzero(keep)


def loader(dataset, indices=None, train=False):
    kept = dataset if indices is None else Subset(dataset, indices)
    return DataLoader(kept, batch_size=BATCH_SIZE if train else 128, shuffle=train, drop_last=train)


train_set, val_set = WithConditions("train", augment=True), WithConditions("val", augment=False)
BINS = list(zip(QP_EDGES[:-1], QP_EDGES[1:]))
for name, dataset in (("train", train_set), ("val", val_set)):
    print(f"{name}: {len(dataset)} patches, " + ", ".join(
        f"{encoder} {len(dataset.where(encoder))} ({len(dataset.where(encoder, intra=True))} I)" for encoder in ENCODERS))

## 2. Models

`Told` hands a `models.ToldLuma` the conditions it is told; `Blind` drops them all. Notebook 19's model is told the
slice QP alone, and notebook 8's graph is read back from its ONNX file as notebook 19 did.

In [ ]:
class Blind(nn.Module):
    """A model that is not told, on loaders that tell."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x, condition=None):
        return self.model(x)


class Told(nn.Module):
    """A `ToldLuma` given the `parts` of `CONDITIONS` it is told."""

    def __init__(self, model, parts):
        super().__init__()
        self.model, self.parts = model, [CONDITIONS.index(part) for part in parts]

    def forward(self, x, condition):
        return self.model(x, condition[:, self.parts])


def build(parts):
    if not parts:
        return Blind(models.LumaUpscaler((WIDTH,), BLOCKS, STEM, scale=SCALE))
    return Told(models.ToldLuma(conditions=len(parts), width=WIDTH, blocks=BLOCKS, stem=STEM, scale=SCALE), parts)


def from_graph(path):
    """An exported `LumaUpscaler` graph as the torch model again: its initializers are the state dict."""
    weights = {t.name: torch.from_numpy(numpy_helper.to_array(t).copy()) for t in onnx.load(str(path)).graph.initializer}
    model = models.LumaUpscaler((WIDTH,), BLOCKS, STEM, scale=SCALE)
    model.load_state_dict(weights)
    return model.to(DEVICE).eval()


def notebook_19():
    saved = training.load_checkpoint(NOTEBOOK_19)
    model = models.ToldLuma(scale=saved["scale"], **saved["arguments"]).to(DEVICE)
    model.load_state_dict(saved["state_dict"])
    return Told(model, ("qp",)).eval()


assert build(CONDITIONS).model.halo <= manifest["halo"], "the model reaches further than the patches' halo"
references = {"notebook 8": Blind(from_graph(SHIPPED)), "notebook 19": notebook_19()}

## 3. Train

Notebook 19's recipe: Adam from 1e-3, batch 32, `metrics.ssim_l1_loss`, the epoch with the best SSIM-Y on all the
validation patches kept, until it has converged. The seed sets the starting weights and then the batches. Run the cell
again after an interruption and it carries on; a finished training just loads.

In [ ]:
def train(name, parts, seed):
    torch.manual_seed(seed)
    model = build(parts).to(DEVICE)
    torch.manual_seed(seed)
    run = f"{name.replace(' + ', '_').replace(' ', '_')}_seed{seed}"
    history, best = training.train(model, loader(train_set, train=True), loader(val_set), trim, MAX_EPOCHS,
                                   learning_rate=LEARNING_RATE, loss_fn=metrics.ssim_l1_loss, select="ssim",
                                   checkpoint=RUN_DIR / f"{run}.pt", verbose=0)
    print(f"{run:<26} best epoch {best['epoch']:3} of {len(history):3}, ssim-y {best['score']:.4f}")
    return model.eval(), best


trained, bests = {}, {}
for seed in (0, 1):
    for name, (parts, seeds) in RUNS.items():
        if seed in seeds:
            trained[name, seed], bests[name, seed] = train(name, parts, seed)

## 4. Gain over bicubic

Each cell is SSIM-Y and luma dB over bicubic. First by encoder and frame type over every slice QP, then by encoder and
slice QP bin on P frames. The told models are the mean of their two seeds; the last column is how far apart the two seeds
of "told QP + frame" land.

In [ ]:
class Bicubic(nn.Module):
    def forward(self, x, condition=None):
        return models.bicubic(x, SCALE)


columns = {**references, "blind": trained["blind", 0],
           **{name: [trained[name, seed] for seed in RUNS[name][1]] for name in ("told QP", "told QP + frame")}}
GROUPS = [("all", {}), *[(f"{e} P", {"encoder": e, "intra": False}) for e in ENCODERS],
          *[(f"{e} I", {"encoder": e, "intra": True}) for e in ENCODERS]]
ROWS = GROUPS + [(f"{e} P {low}-{high}", {"encoder": e, "intra": False, "low": low, "high": high})
                 for e in ENCODERS for low, high in BINS]


def scores_of(model_or_seeds, data):
    """`training.evaluate`'s SSIM-Y and luma dB, the mean over seeds for a list, and the gap between the first two."""
    runs = model_or_seeds if isinstance(model_or_seeds, list) else [model_or_seeds]
    each = [training.evaluate(model, data, trim) for model in runs]
    mean = {key: np.mean([s[key] for s in each]) for key in ("ssim", "y")}
    return mean | {"gap": abs(each[0]["ssim"] - each[1]["ssim"]) if len(each) > 1 else 0.0}


table = {}
for label, where in ROWS:
    indices = val_set.where(**where)
    if len(indices) == 0:
        continue
    data = loader(val_set, indices)
    table[label] = {"patches": len(indices), "bicubic": scores_of(Bicubic(), data),
                    **{name: scores_of(model, data) for name, model in columns.items()}}


def print_table(labels):
    print(f"{'':<16}{'patches':>8}{'bicubic':>9}" + "".join(f"{name:>18}" for name in columns) + f"{'seed gap':>10}")
    for label in labels:
        if label not in table:
            continue
        row, base = table[label], table[label]["bicubic"]
        cells = [f"{row[name]['ssim'] - base['ssim']:+.4f} / {row[name]['y'] - base['y']:+.2f}" for name in columns]
        print(f"{label:<16}{row['patches']:8}{base['ssim']:9.4f}" + "".join(f"{cell:>18}" for cell in cells)
              + f"{row['told QP + frame']['gap']:10.4f}")


print_table([label for label, _ in GROUPS])
print()
print_table([label for label, _ in ROWS[len(GROUPS):]])

## 5. Save

The better seed of "told QP + frame", with what a later notebook needs to rebuild it. Rebuilt from the file alone and
baked for a frame, it must compute what the trained model does. The client's side of it is the line the biases are on:
`b + W c`, with `c` = (QP / 51, loop filter, I frame).

In [ ]:
NAME = "told QP + frame"
best_seed = max(RUNS[NAME][1], key=lambda seed: bests[NAME, seed]["score"])
best = trained[NAME, best_seed].model
training.save_checkpoint(CHECKPOINT, best, architecture="ToldLuma",
                         arguments={"conditions": len(CONDITIONS), "width": WIDTH, "blocks": BLOCKS, "stem": STEM},
                         told=list(CONDITIONS), qp_max=QP_MAX, qp="slice header", scale=SCALE, halo=best.halo,
                         loss="ssim_l1_loss", select="ssim", seed=best_seed, best_epoch=bests[NAME, best_seed]["epoch"],
                         encoders=list(ENCODERS),
                         val={label: {name: {"ssim": float(row[name]["ssim"]), "y": float(row[name]["y"])}
                                      for name in ["bicubic", *columns]} for label, row in table.items()})

saved = training.load_checkpoint(CHECKPOINT)
loaded = models.ToldLuma(scale=saved["scale"], **saved["arguments"]).to(DEVICE)
loaded.load_state_dict(saved["state_dict"])
for frame in ((30 / QP_MAX, 0, 0), (36 / QP_MAX, 1, 1)):
    training.assert_same_model(best.baked(frame), loaded.baked(frame))
print("reloaded model is identical")

## 6. What it says

The run, all as SSIM-Y / luma dB over bicubic on the 9,628 validation patches; told models are the mean of two seeds:

| | notebook 8 (ships) | notebook 19 | blind | told QP | told QP + frame | seed gap |
|---|---|---|---|---|---|---|
| all | +0.0141 / +0.95 | +0.0222 / +1.41 | +0.0211 / +1.16 | +0.0237 / +1.50 | **+0.0242 / +1.53** | 0.0001 |
| x264 P | +0.0148 / +0.79 | +0.0302 / +1.54 | +0.0266 / +1.20 | +0.0300 / +1.54 | +0.0302 / +1.58 | 0.0000 |
| NVENC P | +0.0135 / +1.05 | +0.0140 / +1.22 | +0.0155 / +1.08 | +0.0175 / +1.44 | **+0.0185 / +1.47** | 0.0002 |
| x264 I | +0.0138 / +1.07 | +0.0288 / +1.72 | +0.0258 / +1.46 | +0.0288 / +1.70 | +0.0287 / +1.67 | 0.0002 |
| NVENC I | +0.0136 / +1.14 | +0.0155 / +1.38 | +0.0154 / +1.10 | +0.0170 / +1.47 | +0.0170 / +1.43 | 0.0002 |

1. **Telling the slice QP works as well as telling x264's mean did.** On x264's P frames the model told the slice QP
   gains +0.0300, notebook 19 (told the mean, trained on x264 alone) +0.0302. The label the client can read costs
   nothing, and learning NVENC beside x264 cost x264 nothing either.
2. **The frame flags are worth having, and the gain is NVENC's.** "Told QP + frame" over "told QP" is +0.0005 overall,
   five times the seed gap, and +0.0010 on NVENC's P frames, +0.0013 to +0.0019 a bin from QP 26 to 42 against seed gaps
   of 0.0001 to 0.0004. On x264 it is +0.0001 to +0.0011, inside the noise for most bins.
3. **Told the QP alone, the model is misled on NVENC.** At NVENC QP 30-38 "told QP" scores below blind (-0.0005 and
   -0.0004): the same slice QP means a sharper, deblocked picture from NVENC and a blockier one from x264, and the model
   told only the number blends the two. Adding the loop-filter flag puts it 0.0011 to 0.0015 above blind there.
   Notebook 19, which never saw NVENC, does worse still: below blind from QP 26 up and below notebook 8 in dB from 30 up.
4. **The I-frame flag adds nothing measurable.** On I frames the two told models are within 0.0001 of each other on
   both encoders. It stays in the model since it is free to read and to bake, but it is not why the frame variant wins.
5. **In this data the loop-filter flag is the encoder.** Every x264 frame is unfiltered and every NVENC frame filtered,
   as the host's lines are, so the flag cannot tell "filtered" from "NVENC". AMF, VideoToolbox and the browser host's
   WebCodecs encoder all keep the loop filter on and will be read as NVENC-like; whether they behave like it is untested.
6. **Against what ships:** +0.0242 / +1.53 dB over bicubic, against notebook 8's +0.0141 / +0.95 - another +0.0101
   SSIM-Y and +0.58 dB on the same patches, for 72 biases a frame computed from three numbers the client already reads.

Weak spots: QP 46-52 on NVENC has 64 patches and a seed gap of 0.0011, and there the told models trail blind in dB
(+0.41 against +0.55). The top bins of both encoders are thin; a run with more low-bitrate pairs would firm them up.

**Next:** `checkpoints/upscale_qp_slice.pt` is the model to export. The client side is a bias line per frame from
`stream-h264.js`'s `{qp, isDeblocked, type}`; the graph keeps the blind one's shape with its biases as inputs.